# Public-data feasibility audit

Executed audit of the fixed 30-game 2024 sample. This notebook preserves pitch and nonpitch feed events, audits timestamp structure and runner-state reconstruction, and constructs candidate **event-start intervals**. It does not treat feed timestamps as validated ball-release times, fit a runner-threat model, or inspect threat associations.


## Definitions used here

- An event key is `(game_pk, at_bat_index, event_index)`.
- An event-start interval is the difference between consecutive pitch-event `startTime` values within one plate appearance. Intervening feed actions are retained and flagged.
- A reconstruction opportunity begins after a pitch resolves with first base occupied and second/third empty, when a later pitch or later nonpitch action remains in that plate appearance. This audit definition is not the final modeling denominator.
- Raw and full derived event tables remain in ignored `data/raw/` and `data/processed/`; compact reviewed results are written to `results/`.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import platform
import sys

import jupyterlab
import nbconvert
import nbformat
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.audit_feeds import (
    action_label_inventory, build_game_quality, build_intervals,
    build_opportunities, extract_events, load_feeds,
    representative_trace_candidates, timestamp_format_inventory,
)

executed_utc = datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z')
versions = pd.DataFrame([
    {'package': 'Python', 'version': platform.python_version()},
    {'package': 'pandas', 'version': pd.__version__},
    {'package': 'numpy', 'version': np.__version__},
    {'package': 'requests', 'version': requests.__version__},
    {'package': 'jupyterlab', 'version': jupyterlab.__version__},
    {'package': 'nbconvert', 'version': nbconvert.__version__},
    {'package': 'nbformat', 'version': nbformat.__version__},
])
print(f'Executed UTC: {executed_utc}')
display(versions)


Executed UTC: 2026-11-15T20:40:45Z

,package,version
0,Python,3.14.4
1,pandas,3.0.2
2,numpy,2.4.4
3,requests,2.34.0
4,jupyterlab,4.6.1
5,nbconvert,7.17.1
6,nbformat,5.10.4


## Fixed schedule-only sample and cached feeds

The selection script uses a fixed seed and schedule metadata only: 10 games from each of early (March 28-April 30), middle (June 15-July 15), and late (September 1-29) 2024. Within each window it accepts the seeded order subject to one game per date and venue. No live-feed timing or running outcome is used in selection.


In [2]:
manifest, loaded_feeds = load_feeds(ROOT)
coverage = (manifest.groupby('period', sort=False)
    .agg(games=('game_pk', 'size'), first_date=('official_date', 'min'),
         last_date=('official_date', 'max'), venues=('venue', 'nunique'))
    .reset_index())
display(coverage)
all_teams = pd.unique(pd.concat([manifest['away_team'], manifest['home_team']]))
print(f"Total: {len(manifest)} games, {manifest['venue'].nunique()} venues, {len(all_teams)} teams")
print('Feed failures:', int(manifest['download_status'].eq('failed').sum()))
assert len(manifest) == len(loaded_feeds) == 30
assert manifest['sha256'].str.fullmatch(r'[0-9a-f]{64}').all()


,period,games,first_date,last_date,venues
0,early,10,2024-04-01,2024-04-29,10
1,middle,10,2024-06-16,2024-07-13,10
2,late,10,2024-09-07,2024-09-29,10


Total: 30 games, 21 venues, 24 teams

Feed failures:

0

## Extract all events and reconstruct candidate states

Runner movements are applied by `details.playIndex`. Multi-segment paths for one runner are collapsed from the pre-event origin to the last destination. `runner_placed` and pinch-runner actions update occupancy explicitly. Half-innings are cleared on a recorded third out. No event is dropped because of duration or label.


In [3]:
events, state_audit = extract_events(loaded_feeds)
intervals = build_intervals(events)
opportunities = build_opportunities(events)
game_quality = build_game_quality(events, intervals, state_audit)
action_inventory = action_label_inventory(events)
timestamp_inventory = timestamp_format_inventory(events)
trace_candidates = representative_trace_candidates(opportunities)

pa_count = events.groupby('game_pk')['at_bat_index'].nunique().sum()
print(f"{events['game_pk'].nunique():,} games; {pa_count:,} plate appearances")
print(f"{len(events):,} events: {events['is_pitch'].sum():,} pitches and {(~events['is_pitch']).sum():,} nonpitch events")
print(f"{len(intervals):,} candidate event-start intervals; {len(opportunities):,} first-base-only audit opportunities")
assert len(events) == game_quality['events'].sum()
assert not events.duplicated(['game_pk', 'at_bat_index', 'event_index']).any()


30 games; 2,283 plate appearances

10,156 events: 8,808 pitches and 1,348 nonpitch events

6,532 candidate event-start intervals; 1,253 first-base-only audit opportunities

## Missingness and anomalies by game

All 30 games are shown; zeroes are findings, not suppressed rows. `within_pa_nonpositive_start_deltas` concerns adjacent feed events of any type. Pitch-to-pitch candidate intervals are reported separately.


In [4]:
quality_columns = [
    'game_pk', 'official_date', 'events', 'pitches', 'nonpitch_events',
    'missing_start_time', 'invalid_start_time', 'missing_end_time', 'invalid_end_time',
    'duplicate_event_keys', 'nonincreasing_event_indexes',
    'within_pa_negative_start_deltas', 'within_pa_nonpositive_start_deltas',
    'nonpositive_event_start_intervals', 'unresolved_runner_movements',
    'runner_movement_anomalies', 'matchup_end_state_mismatches',
]
with pd.option_context('display.max_rows', 35, 'display.max_columns', None):
    display(game_quality[quality_columns])


,game_pk,official_date,events,pitches,nonpitch_events,missing_start_time,invalid_start_time,missing_end_time,invalid_end_time,duplicate_event_keys,nonincreasing_event_indexes,within_pa_negative_start_deltas,within_pa_nonpositive_start_deltas,nonpositive_event_start_intervals,unresolved_runner_movements,runner_movement_anomalies,matchup_end_state_mismatches
0,746409,2024-04-01,248,220,28,0,0,0,0,0,0,0,0,0,0,0,0
1,746900,2024-04-03,382,330,52,0,0,0,0,0,0,0,0,0,0,0,0
2,745764,2024-04-05,375,326,49,0,0,0,0,0,0,0,0,0,0,0,0
3,746493,2024-04-07,339,299,40,0,0,0,0,0,0,0,0,0,0,0,0
4,746655,2024-04-10,376,293,83,0,0,0,0,0,0,0,0,0,0,0,0
5,744944,2024-04-16,361,325,36,0,0,0,0,0,0,0,0,0,0,0,0
6,746572,2024-04-21,322,290,32,0,0,0,0,0,0,0,0,0,0,0,0
7,745919,2024-04-24,334,284,50,0,0,0,0,0,0,0,1,0,0,0,0
8,745027,2024-04-25,313,270,43,0,0,0,0,0,0,0,0,0,0,0,0
9,747211,2024-04-29,345,312,33,0,0,0,0,0,0,0,0,0,0,0,0


## Timestamp format, ordering, and interval structure

Three fractional digits demonstrate millisecond *formatting* only. They do not document capture accuracy, clock alignment, or whether `startTime` is release time. Nonpositive adjacent event differences are retained for review; no candidate pitch-to-pitch interval is filtered.


In [5]:
display(timestamp_inventory)
ordered = events.sort_values(['game_pk', 'play_order', 'event_order']).copy()
ordered['within_pa_start_delta_seconds'] = (
    ordered.groupby(['game_pk', 'at_bat_index'])['start_time'].diff().dt.total_seconds())
nonpositive_events = ordered.loc[ordered['within_pa_start_delta_seconds'].le(0), [
    'game_pk', 'at_bat_index', 'event_index', 'event_record_type', 'event_type',
    'description', 'start_time_raw', 'within_pa_start_delta_seconds']]
display(nonpositive_events)
interval_stats = intervals['event_start_interval_seconds'].describe(
    percentiles=[0.01, 0.5, 0.95, 0.99]).rename('seconds').to_frame()
display(interval_stats)
flag_columns = ['pickoff_flag', 'step_off_flag', 'timeout_flag', 'mound_visit_flag',
                'substitution_flag', 'other_nonpitch_flag',
                'pitcher_changed_flag', 'batter_changed_flag']
interval_flags = intervals[flag_columns].sum().astype(int).rename('intervals').to_frame()
interval_flags.loc['over_60_seconds'] = intervals['event_start_interval_seconds'].gt(60).sum()
display(interval_flags)


,field,fractional_second_digits,events,example
0,startTime,3,10156,2024-04-21T23:24:59.056Z
1,endTime,3,10156,2024-04-21T23:47:42.573Z


,game_pk,at_bat_index,event_index,event_record_type,event_type,description,start_time_raw,within_pa_start_delta_seconds
7057,745368,66,2,action,defensive_substitution,Defensive Substitution: Patrick Bailey replace...,2024-09-08T22:33:15.352Z,0.0
5190,745883,57,3,action,pitching_substitution,Pitching Change: Griffin Jax replaces Josh Sta...,2024-07-07T20:12:15.912Z,0.0
2288,745919,46,2,action,defensive_switch,Max Kepler remains in the game as the right fi...,2024-04-25T01:14:08.944Z,0.0
5570,746043,63,2,action,pitching_substitution,Pitching Change: Kenley Jansen replaces Brenna...,2024-07-04T19:36:25.488Z,0.0
5582,746043,66,3,no_pitch,NaN,Automatic Ball - Intentional,2024-07-04T19:44:19.767Z,0.0
5657,746043,78,3,no_pitch,NaN,Automatic Ball - Intentional,2024-07-04T20:10:43.753Z,0.0
5707,746043,91,3,no_pitch,NaN,Automatic Ball - Intentional,2024-07-04T20:33:33.924Z,0.0


,seconds
count,6532.000000
mean,20.081041
std,8.855531
min,8.033000
1%,11.110340
50%,18.356000
95%,33.012250
99%,49.144200
max,290.200000


,intervals
pickoff_flag,72
step_off_flag,43
timeout_flag,377
mound_visit_flag,13
substitution_flag,2
other_nonpitch_flag,70
pitcher_changed_flag,1
batter_changed_flag,0
over_60_seconds,33


## Interruption and nonpitch label inventory

This is an inventory of recorded labels, not proof that unlabeled intervals were uninterrupted. Pickoff attempts and step-offs often have a record `type` but no `details.eventType`; both fields must be retained.


In [6]:
display(action_inventory.groupby('action_category', as_index=False)['events'].sum())
with pd.option_context('display.max_rows', 40):
    display(action_inventory)


,action_category,events
0,mound_visit,176
1,other_nonpitch,226
2,pickoff,100
3,step_off,57
4,substitution,377
5,timeout,412


,action_category,event_record_type,event_type_display,event_name_display,events
0,mound_visit,action,mound_visit,Mound Visit,176
1,other_nonpitch,action,game_advisory,Game Advisory,119
2,other_nonpitch,no_pitch,(missing),(missing),36
3,other_nonpitch,action,wild_pitch,Wild Pitch,21
4,other_nonpitch,action,stolen_base_2b,Stolen Base 2B,19
5,other_nonpitch,action,runner_placed,Runner Placed On Base,10
6,other_nonpitch,action,caught_stealing_2b,Caught Stealing 2B,5
7,other_nonpitch,action,ejection,Ejection,5
8,other_nonpitch,action,stolen_base_3b,Stolen Base 3B,5
9,other_nonpitch,action,passed_ball,Passed Ball,3


## State semantics and reconstruction checks

The feed's `matchup.splits.menOnBase` is not a safe pre-action predictor. In this sample it agrees with the reconstructed end-of-plate-appearance occupancy bucket in every play, while it often differs from occupancy before the first pitch. Event `count` values likewise need event-specific interpretation: balls/strikes update on ordinary pitches, but a terminal force-play example retains two outs even though runner movement records `outNumber = 3`.


In [7]:
state_summary = pd.Series({
    'plate_appearances': len(state_audit),
    'missing_first_pitch_state': state_audit['reconstructed_before_first_pitch'].isna().sum(),
    'matchup_matches_pre_first_pitch': state_audit['matchup_matches_start'].sum(),
    'matchup_matches_pa_end': state_audit['matchup_matches_end'].sum(),
    'unresolved_movement_records': state_audit['unresolved_movement_count'].sum(),
    'movement_anomalies': state_audit['movement_anomaly_count'].sum(),
}).rename('count').to_frame()
display(state_summary)
state_issues = state_audit.loc[
    state_audit['movement_anomaly_count'].gt(0) | ~state_audit['matchup_matches_end']].copy()
display(state_issues[[
    'game_pk', 'at_bat_index', 'inning', 'half_inning',
    'reconstructed_pa_start_bases', 'reconstructed_pa_end_bases',
    'matchup_men_on_base', 'play_result_event_type', 'movement_anomalies']])
terminal_example = events.loc[
    events['game_pk'].eq(746690) & events['at_bat_index'].eq(38),
    ['event_index', 'description', 'derived_pre_outs', 'feed_count_outs', 'bases_before',
     'bases_after', 'movement_summary', 'movement_anomaly']]
display(terminal_example)


,count
plate_appearances,2283
missing_first_pitch_state,7
matchup_matches_pre_first_pitch,1334
matchup_matches_pa_end,2283
unresolved_movement_records,0
movement_anomalies,1


,game_pk,at_bat_index,inning,half_inning,reconstructed_pa_start_bases,reconstructed_pa_end_bases,matchup_men_on_base,play_result_event_type,movement_anomalies
1479,746690,38,4,bottom,1B+2B,empty,Empty,force_out,occupied destination 1B: 668715 replaced by 68...


,event_index,description,derived_pre_outs,feed_count_outs,bases_before,bases_after,movement_summary,movement_anomaly
6561,0,Ball,2,2,1B+2B,1B+2B,,
6562,1,"In play, out(s)",2,2,1B+2B,1B,Rece Hinds: 2B->out@3B (force_out) | Elly De L...,occupied destination 1B: 668715 replaced by 68...


## Opportunity reconstruction audit

Ordinary nonattempts remain in the denominator. Recorded pickoffs, steals, caught stealing, and a candidate opportunity ending without another pitch are retained. The trace candidates below are identifiers for manual comparison with MLB's public Gameday play-by-play; reviewed outcomes are recorded separately in `results/manual_play_by_play_checks.csv`.


In [8]:
opportunity_summary = (opportunities.groupby(['status', 'opportunity_class'])
    .size().rename('opportunities').reset_index())
display(opportunity_summary)
display(trace_candidates[[
    'trace_case', 'game_pk', 'official_date', 'at_bat_index',
    'opportunity_event_index', 'next_pitch_event_index', 'feed_event_count',
    'intervening_actions', 'next_pitch_movement', 'play_result']])


,status,opportunity_class,opportunities
0,next_pitch,caught_stealing,9
1,next_pitch,ordinary_nonattempt,1171
2,next_pitch,pickoff,56
3,next_pitch,stolen_base,16
4,no_next_pitch_after_later_action,pickoff,1


,trace_case,game_pk,official_date,at_bat_index,opportunity_event_index,next_pitch_event_index,feed_event_count,intervening_actions,next_pitch_movement,play_result
0,ordinary nonattempt,746409,2024-04-01,1,0,1.0,0-1,,,Vladimir Guerrero Jr. strikes out swinging.
1,pickoff with later pitch,746900,2024-04-03,10,2,4.0,2-1,3:nan (Pickoff Attempt 1B),,Dansby Swanson singles on a line drive to left...
2,recorded stolen base,746900,2024-04-03,12,4,6.0,3-1,5:stolen_base_2b (Dansby Swanson steals (1) 2n...,,Nico Hoerner walks.
3,recorded caught stealing,745764,2024-04-05,14,3,5.0,2-1,4:caught_stealing_2b (Anthony Volpe caught ste...,,Alex Verdugo singles on a ground ball to third...
4,opportunity ending without another pitch,746655,2024-04-10,13,4,NaN,1-2,5:nan (Pickoff Attempt 1B) | 6:nan (Pickoff At...,,Pitcher Tanner Bibee picks off Dominic Fletche...


## Save reproducible outputs

Full derived tables remain ignored because they can be regenerated from cached raw feeds. Compact audit tables are reviewed deliverables.


In [9]:
processed = ROOT / 'data' / 'processed'
results = ROOT / 'results'
processed.mkdir(parents=True, exist_ok=True)
results.mkdir(parents=True, exist_ok=True)
events.to_csv(processed / 'audit_events.csv', index=False)
intervals.to_csv(processed / 'audit_event_start_intervals.csv', index=False)
opportunities.to_csv(processed / 'audit_opportunities.csv', index=False)
state_audit.to_csv(processed / 'audit_pa_states.csv', index=False)
game_quality.to_csv(results / 'audit_game_quality.csv', index=False)
action_inventory.to_csv(results / 'audit_action_label_inventory.csv', index=False)
interval_stats.reset_index(names='statistic').to_csv(results / 'audit_interval_summary.csv', index=False)
opportunity_summary.to_csv(results / 'audit_opportunity_summary.csv', index=False)
state_issues.to_csv(results / 'audit_state_issues.csv', index=False)
trace_candidates.to_csv(results / 'audit_trace_candidates.csv', index=False)
versions.assign(executed_utc=executed_utc).to_csv(results / 'audit_environment.csv', index=False)
print('Wrote 4 ignored full derived tables and 7 compact reviewed result tables.')


Wrote 4 ignored full derived tables and 7 compact reviewed result tables.

## Measurement boundary

This notebook completes automated structural checks and prepares manual play-by-play checks. It performs **no video timing validation**. Until release events are observed on public video under the prerecorded acceptance rule, these values remain event-start intervals, and absence of a feed interruption label cannot establish uninterrupted play.
